# Практикум 18. Метрики качества: BLEU, ROUGE и F1 на практике

В лаборатории 18 вы считали BLEU, ROUGE и F1 по шагам на готовых примерах. Здесь напишем все метрики с нуля и сверим их с библиотеками:

1. BLEU: n-граммы, отсечение (clipping), штраф за краткость, среднее геометрическое; сверка с `nltk.translate.bleu_score` при одинаковом сглаживании; BLEU предложения и BLEU корпуса;
2. chrF — F-мера по символьным n-граммам — и почему она мягче к ошибкам в узбекских суффиксах;
3. ROUGE-1, ROUGE-2 и ROUGE-L через таблицу динамического программирования для LCS;
4. точность, полнота и F1 для именованных сущностей (по сущностям и по токенам), micro- и macro-усреднение со сверкой по `sklearn.metrics`.

*Практикум к лаборатории 18 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. BLEU по шагам

BLEU сравнивает перевод системы с одним или несколькими эталонными переводами по совпадающим n-граммам, n = 1..4.

- Модифицированная точность: $p_n = \dfrac{\sum_g \min\big(\text{count}_{hyp}(g),\ \max_{ref} \text{count}_{ref}(g)\big)}{\sum_g \text{count}_{hyp}(g)}$, где $g$ — n-граммы перевода. Отсечение (clipping, $\min$) не даёт «накрутить» точность повтором слова.
- Штраф за краткость: $BP = 1$, если $c > r$, иначе $BP = e^{\,1 - r/c}$; $c$ — длина перевода, $r$ — длина эталона, ближайшего к переводу по длине.
- $\text{BLEU} = BP \cdot \exp\big(\tfrac14 \sum_{n=1}^{4} \log p_n\big)$, то есть штраф, умноженный на среднее геометрическое четырёх точностей.

Примеры взяты из лаборатории. Проследим расчёт для перевода, где одно слово заменено синонимом: эта замена портит 1 униграмму, 2 биграммы, 3 триграммы и 4 четырёхграммы.

Во второй половине ячейки видно, зачем нужно отсечение. Перевод из восьми «мы» без отсечения получает $p_1 = 1$: каждое «мы» есть в эталоне. С отсечением «мы» засчитывается не больше раз, чем встречается в эталоне, то есть один раз.

In [ ]:
import re, math, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ref1 = "Вчера вечером мы долго гуляли в старом парке."
ref2 = "Мы вчера вечером долго гуляли по старому парку."
cands = {
    "Копия": "Вчера вечером мы долго гуляли в старом парке.",
    "Синоним": "Вчера вечером мы долго бродили в старом парке.",
    "Порядок слов": "В старом парке мы вчера вечером долго гуляли.",
    "Окончание": "Вчера вечером мы долго гуляем в старом парке.",
    "Короткий": "Мы долго гуляли в старом парке.",
    "Повторы": "мы мы мы мы мы мы мы мы",
}

def tokenize(s):
    # нижний регистр, без знаков препинания; ʻ и ʼ остаются внутри слова
    return re.findall(r"\w+(?:[-'’]\w+)*", s.lower())

def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def clipped_counts(hyp, refs, n, clip=True):
    hyp_counts = Counter(ngrams(hyp, n))
    max_ref = Counter()                       # сколько раз n-грамма встречается в «самом щедром» эталоне
    for r in refs:
        for g, c in Counter(ngrams(r, n)).items():
            max_ref[g] = max(max_ref[g], c)
    credit = {g: (min(c, max_ref[g]) if clip else (c if max_ref[g] else 0)) for g, c in hyp_counts.items()}
    return hyp_counts, max_ref, credit

def closest_ref_len(refs, c):
    return min((abs(len(r) - c), len(r)) for r in refs)[1]   # при равенстве — более короткий

def brevity_penalty(c, r):
    if c > r:
        return 1.0
    return 0.0 if c == 0 else math.exp(1 - r / c)

def trace(hyp_text, ref_texts, clip=True, max_n=4):
    hyp, refs = tokenize(hyp_text), [tokenize(r) for r in ref_texts]
    ps = []
    for n in range(1, max_n + 1):
        hyp_counts, max_ref, credit = clipped_counts(hyp, refs, n, clip)
        for g, c in hyp_counts.items():
            mark = "" if credit[g] else "   ✗"
            print(f"   {' '.join(g):<36} в переводе {c}, в эталоне {max_ref[g]} -> засчитано {credit[g]}{mark}")
        matched, total = sum(credit.values()), max(sum(hyp_counts.values()), 1)
        ps.append(matched / total)
        print(f"p{n} = {matched}/{total} = {matched / total:.3f}\n")
    c, r = len(hyp), closest_ref_len(refs, len(hyp))
    bp = brevity_penalty(c, r)
    geo = math.exp(sum(math.log(p) for p in ps) / max_n) if min(ps) > 0 else 0.0
    name = "BLEU" if max_n == 4 else f"BLEU (n ≤ {max_n})"
    print(f"c = {c}, r = {r}, BP = {bp:.3f};  среднее геометрическое = {geo:.3f};  {name} = {bp * geo:.3f}")

print(f"=== «{cands['Синоним']}»")
trace(cands["Синоним"], [ref1])

print()

for clip, name in [(False, "без отсечения"), (True, "с отсечением")]:
    print(f"=== {name}: «{cands['Повторы']}»")
    trace(cands["Повторы"], [ref1], clip=clip, max_n=1)
    print()

Теперь функция `bleu` целиком и сверка с `nltk.translate.bleu_score.sentence_bleu` для всех примеров. Без сглаживания одна нулевая $p_n$ обнуляет весь BLEU (логарифм нуля). Сглаживание сравниваем при одинаковых настройках у нас и в nltk:

- `SmoothingFunction().method1`: если совпадений n-го порядка нет, вместо 0 берётся $\varepsilon / \text{знаменатель}$, $\varepsilon = 0{,}1$;
- `SmoothingFunction().method2`: к числителю и знаменателю $p_n$ при $n \ge 2$ прибавляется 1 (галочка «Сглаживание» в лаборатории).

Строка под таблицей — наибольшее отличие нашей функции от nltk по всем примерам и настройкам (порядка $10^{-16}$ или меньше, то есть совпадение до ошибок округления).

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

def bleu(hyp, refs, smoothing=None, eps=0.1, max_n=4):
    """hyp — список токенов, refs — список таких списков; smoothing: None, "eps" (nltk method1), "add1" (nltk method2)"""
    matched, total = [], []
    for n in range(1, max_n + 1):
        hyp_counts, _, credit = clipped_counts(hyp, refs, n)
        matched.append(sum(credit.values()))
        total.append(max(sum(hyp_counts.values()), 1))   # как в nltk: знаменатель не меньше 1
    if matched[0] == 0:
        return 0.0
    ps = []
    for n, (m, t) in enumerate(zip(matched, total), start=1):
        if smoothing == "add1" and n > 1:
            ps.append((m + 1) / (t + 1))
        elif smoothing == "eps" and m == 0:
            ps.append(eps / t)
        else:
            ps.append(m / t)
    if min(ps) == 0:
        return 0.0
    bp = brevity_penalty(len(hyp), closest_ref_len(refs, len(hyp)))
    return bp * math.exp(sum(math.log(p) for p in ps) / max_n)

NLTK_SMOOTHING = {None: SmoothingFunction().method0, "eps": SmoothingFunction().method1, "add1": SmoothingFunction().method2}
R1, R2 = tokenize(ref1), tokenize(ref2)
rows, max_diff = [], 0.0
for name, text in cands.items():
    h = tokenize(text)
    row = {"пример": name}
    for n in range(1, 5):
        _, _, cr = clipped_counts(h, [R1], n)
        row[f"p{n}"] = sum(cr.values()) / max(len(ngrams(h, n)), 1)
    row["BP"] = brevity_penalty(len(h), closest_ref_len([R1], len(h)))
    for col, refs, sm in [("BLEU", [R1], None), ("2 эталона", [R1, R2], None),
                          ("method1", [R1], "eps"), ("method2", [R1], "add1"), ("2 + method2", [R1, R2], "add1")]:
        ours = bleu(h, refs, sm)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")      # nltk предупреждает, когда какая-то p_n = 0
            ref_value = sentence_bleu(refs, h, smoothing_function=NLTK_SMOOTHING[sm])
        max_diff = max(max_diff, abs(ours - ref_value))
        row[col] = ours
    rows.append(row)
print(pd.DataFrame(rows).set_index("пример").round(3).to_string())
print(f"\nнаибольшее расхождение с nltk: {max_diff:.1e}")

### BLEU предложения и BLEU корпуса

BLEU придуман для целого тестового корпуса. Корпусный BLEU сначала складывает по всем предложениям числители и знаменатели $p_n$, а также длины $c$ и $r$, и только потом делит (micro-усреднение). Среднее BLEU отдельных предложений — другое число: без сглаживания короткое предложение легко получает 0, если в нём нет ни одной общей четырёхграммы. Сравним на шести предложениях маленького «тестового набора».

In [ ]:
test_refs = [
    "Поезд из Ташкента прибывает в Самарканд утром.",
    "Библиотека открыта каждый день, кроме воскресенья.",
    "Студенты сдали экзамен по математике.",
    "Завтра днём будет солнечно и тепло, без осадков.",
    "Мой брат работает врачом в городской больнице.",
    "Мы купили на рынке свежие фрукты.",
]
test_hyps = [
    "Поезд из Ташкента приходит в Самарканд утром.",
    "Библиотека работает каждый день, кроме воскресенья.",
    "Студенты сдали экзамен по математике.",
    "Завтра будет солнечно и тепло.",
    "Брат мой работает доктором в больнице города.",
    "Мы купили свежие фрукты на рынке.",
]

hyps = [tokenize(h) for h in test_hyps]
refs_list = [[tokenize(r)] for r in test_refs]

def corpus_bleu_ours(hyps, refs_list, max_n=4):
    matched, total = np.zeros(max_n), np.zeros(max_n)
    c = r = 0
    for hyp, refs in zip(hyps, refs_list):
        for n in range(1, max_n + 1):
            hyp_counts, _, credit = clipped_counts(hyp, refs, n)
            matched[n - 1] += sum(credit.values())     # сначала складываем по всем предложениям...
            total[n - 1] += sum(hyp_counts.values())
        c += len(hyp)
        r += closest_ref_len(refs, len(hyp))
    ps = matched / total                               # ...и только потом делим
    print("p_n =", np.round(ps, 3), f" c = {c}, r = {r}")
    return brevity_penalty(c, r) * math.exp(np.mean(np.log(ps))) if ps.min() > 0 else 0.0

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sent_plain = [sentence_bleu(refs, h) for h, refs in zip(hyps, refs_list)]
    sent_smooth = [sentence_bleu(refs, h, smoothing_function=SmoothingFunction().method2) for h, refs in zip(hyps, refs_list)]
for h, b0, b2 in zip(test_hyps, sent_plain, sent_smooth):
    print(f"{h:<52} BLEU предл. = {b0:.3f}   method2 = {b2:.3f}")
print(f"\nсреднее BLEU предложений: {np.mean(sent_plain):.3f} (method2: {np.mean(sent_smooth):.3f})")
ours = corpus_bleu_ours(hyps, refs_list)
print(f"корпусный BLEU: наш {ours:.4f}, nltk.corpus_bleu {corpus_bleu(refs_list, hyps):.4f}")

Корпусный BLEU не равен среднему BLEU предложений: вполне приличные переводы без единой общей четырёхграммы получили BLEU = 0, но в корпусе их совпадения по-прежнему учитываются. Поэтому в отчётах приводят корпусный BLEU, а BLEU отдельного предложения считают только со сглаживанием.

### chrF: F-мера по символьным n-граммам

Для BLEU «гуляли» и «гуляем» так же различны, как «гуляли» и «стол». chrF (Popović, 2015) убирает пробелы и сравнивает символьные n-граммы, n = 1..6. Для каждого n считаются точность $P_n$ и полнота $R_n$, затем их средние $P$ и $R$, и F-мера, в которой полнота важнее в $\beta = 2$ раза: $\text{chrF} = \dfrac{(1+\beta^2)\,P R}{\beta^2 P + R}$.

В nltk есть `sentence_chrf`, но он усредняет F-меры отдельных порядков $F_n$, а не $P$ и $R$ (и по умолчанию берёт $\beta = 3$). Поэтому наша функция умеет оба способа: способ nltk сверяем с nltk при $\beta = 2$, а стандартный способ (как в sacrebleu и в лаборатории) сравниваем с BLEU. Заметно они расходятся на коротких строках: способ nltk считает $F_n = 0$ для порядка $n$, которого нет в строке, а стандартный такой порядок пропускает. На наших примерах числа почти совпадают.

**Почему chrF подходит узбекскому.** В агглютинативном языке одно грамматическое значение — один суффикс. Ошибка в одном значении (doʻstlarimiz «наши друзья» → doʻstlarim «мои друзья») делает всё слово «чужим» для BLEU, хотя основа совпала. chrF засчитывает общие символьные n-граммы основы и штрафует только за изменившийся хвост. Во второй части ячейки посмотрим на отдельные слова с одной ошибкой в окончании или суффиксе: сначала русские, потом узбекские. Для BLEU каждое такое слово просто не совпало (0), chrF даёт частичную оценку.

In [ ]:
from nltk.translate.chrf_score import sentence_chrf

def char_ngrams(text, n):
    s = re.sub(r"\s+", "", text)                       # пробелы убираем
    return Counter(s[i:i + n] for i in range(len(s) - n + 1))

def chrf(hyp, ref, n_max=6, beta=2.0, average="pr"):
    """average="pr": средние P и R по n, затем F (sacrebleu, лаборатория); average="f": среднее F_n по n (nltk)"""
    b2 = beta ** 2
    Ps, Rs, Fs = [], [], []
    for n in range(1, n_max + 1):
        h, r = char_ngrams(hyp, n), char_ngrams(ref, n)
        if not h or not r:
            Fs.append(0.0)
            continue
        m = sum((h & r).values())                      # & у Counter — поэлементный минимум
        p, rc = m / sum(h.values()), m / sum(r.values())
        Ps.append(p); Rs.append(rc)
        Fs.append((1 + b2) * p * rc / (b2 * p + rc) if p + rc else 0.0)
    if average == "f":
        return float(np.mean(Fs))
    P, R = (np.mean(Ps), np.mean(Rs)) if Ps else (0.0, 0.0)
    return float((1 + b2) * P * R / (b2 * P + R)) if P + R else 0.0

ref_text = " ".join(tokenize(ref1))
rows = []
for name, text in cands.items():
    hyp_text = " ".join(tokenize(text))
    rows.append({"пример": name,
                 "nltk (среднее F по n)": sentence_chrf(ref_text, hyp_text, beta=2.0),
                 "наш, способ nltk": chrf(hyp_text, ref_text, average="f"),
                 "наш, стандартный": chrf(hyp_text, ref_text),
                 "BLEU": bleu(tokenize(text), [tokenize(ref1)])})
table = pd.DataFrame(rows).set_index("пример")
print(table.round(3).to_string())
print(f"\nнаибольшее расхождение с nltk: {(table['nltk (среднее F по n)'] - table['наш, способ nltk']).abs().max():.1e}\n")


import os

word_pairs = [   # (слово эталона, слово перевода): ошибка в одном окончании или суффиксе
    ("гуляли", "гуляем"),
    ("книгами", "книгой"),
    ("библиотеку", "библиотеке"),
    ("doʻstlarimiz", "doʻstlarim"),          # «наши друзья» -> «мои друзья»
    ("kitoblarimizdan", "kitoblarimizda"),    # «из наших книг» -> «в наших книгах»
    ("kutubxonaga", "kutubxonada"),           # «в библиотеку» -> «в библиотеке»
    ("samarqandning", "samarqand"),           # «Самарканда» -> «Самарканд»
    ("uyga", "uyda"),                         # «домой» -> «дома»
]

for ref_word, hyp_word in word_pairs:
    shared = len(os.path.commonprefix([ref_word, hyp_word]))     # длина общей начальной части
    print(f"{ref_word:<16} -> {hyp_word:<16} BLEU: 0   общих букв: {shared:2d} / {len(ref_word):2d}   chrF = {chrf(hyp_word, ref_word):.2f}")

ax = table[["BLEU", "наш, стандартный"]].rename(columns={"наш, стандартный": "chrF"}).plot.bar(figsize=(8, 3.5), color=["#2B55C9", "#17907E"], rot=15)
ax.set_ylabel("оценка"); ax.set_ylim(0, 1.05); ax.set_title("BLEU и chrF для примеров из лаборатории (один эталон)")
plt.tight_layout()
plt.show()

Оценка chrF растёт с длиной общей части: при 4 общих буквах (гуляли → гуляем) она около 0,35, при 14 (kitoblarimizdan → kitoblarimizda) больше 0,9. Узбекские словоформы длинные, а суффиксы присоединяются в конце неизменной основы, поэтому ошибка в одном суффиксе оставляет длинную общую часть. Короткие слова (uyga → uyda) получают мало в любом языке. Обратная сторона: chrF может простить настоящую ошибку смысла, если она выражена коротким суффиксом: kutubxonaga «в библиотеку» и kutubxonada «в библиотеке» — разные места, а оценка 0,77.

### В стандартных инструментах (только Colab)

В статьях по машинному переводу BLEU и chrF считают пакетом `sacrebleu`. В Colab он не установлен: ячейка ставит его и сравнивает с нашими функциями. По умолчанию sacrebleu сам токенизирует текст и учитывает регистр, поэтому для одинаковых чисел передаём ему наши токены в нижнем регистре и `tokenize="none"`.

In [ ]:
# Только для Colab: ставим sacrebleu, стандартный пакет для BLEU и chrF в статьях по машинному переводу
!pip install -q sacrebleu
try:
    import sacrebleu
    hyp_strings = [" ".join(h) for h in hyps]
    ref_strings = [" ".join(rs[0]) for rs in refs_list]
    # тексты уже токенизированы и в нижнем регистре, поэтому tokenize="none" и без сглаживания
    sb = sacrebleu.corpus_bleu(hyp_strings, [ref_strings], tokenize="none", smooth_method="none")
    print(f"BLEU: sacrebleu {sb.score / 100:.4f}   наш {corpus_bleu_ours(hyps, refs_list):.4f}")
    # по умолчанию sacrebleu сам токенизирует (13a) и учитывает регистр: число будет другим
    print("sacrebleu, сырые тексты:", sacrebleu.corpus_bleu(test_hyps, [test_refs]))
    chrf_metric = sacrebleu.CHRF()           # char_order=6, word_order=0, beta=2
    for h, r in zip(hyp_strings, ref_strings):
        print(f"chrF: sacrebleu {chrf_metric.sentence_score(h, [r]).score / 100:.4f}   наш {chrf(h, r):.4f}   {h}")
except Exception as e:
    print("Не получилось (нужен интернет и пакет sacrebleu):", e)

## 2. ROUGE-1, ROUGE-2 и ROUGE-L

ROUGE оценивает автоматические рефераты (тема 13). Как и BLEU, он считает общие n-граммы, но главная величина в нём — полнота: какую долю эталона покрыл реферат.

- ROUGE-N: $P = \dfrac{\text{общих n-грамм}}{\text{n-грамм в реферате}}$, $R = \dfrac{\text{общих n-грамм}}{\text{n-грамм в эталоне}}$, $F_1 = \dfrac{2PR}{P+R}$.
- ROUGE-L: вместо n-грамм берётся наибольшая общая подпоследовательность (LCS) — слова, которые в обоих текстах идут в одном порядке, но не обязательно подряд. Её длину находит динамическое программирование: $T[i][j] = T[i-1][j-1] + 1$, если $ref_i = cand_j$, иначе $T[i][j] = \max(T[i-1][j],\ T[i][j-1])$. Тогда $P = LCS / |cand|$, $R = LCS / |ref|$.

Заполним таблицу для примера «Перестановка» из лаборатории и обратным ходом из правого нижнего угла найдём саму подпоследовательность (на рисунке она выделена).

In [ ]:
rouge_ref = "Мэрия Казани решила построить новую библиотеку в центре города"
summaries = {
    "Перестановка": "Мэрия решила построить в центре Казани новую библиотеку",
    "Перефраз": "В центре Казани появится новая библиотека",
    "Обратный порядок": "города центре в библиотеку новую построить решила Казани мэрия",
    "Длинный": "Вчера мэрия Казани наконец решила построить большую новую библиотеку в самом центре города",
}

def prf(overlap, n_cand, n_ref):
    overlap = int(overlap)
    P = overlap / n_cand if n_cand else 0.0
    R = overlap / n_ref if n_ref else 0.0
    return P, R, (2 * P * R / (P + R) if P + R else 0.0)

def rouge_n(cand, ref, n):
    c, r = Counter(ngrams(cand, n)), Counter(ngrams(ref, n))
    return prf(sum((c & r).values()), sum(c.values()), sum(r.values()))

def lcs_table(ref, cand):
    T = np.zeros((len(ref) + 1, len(cand) + 1), dtype=int)
    for i in range(1, len(ref) + 1):
        for j in range(1, len(cand) + 1):
            if ref[i - 1] == cand[j - 1]:
                T[i, j] = T[i - 1, j - 1] + 1                 # слова совпали: диагональ + 1
            else:
                T[i, j] = max(T[i - 1, j], T[i, j - 1])       # иначе большее из «сверху» и «слева»
    return T

def lcs_backtrack(T, ref, cand):
    i, j, words, path = len(ref), len(cand), [], []
    while i > 0 and j > 0:
        if ref[i - 1] == cand[j - 1]:
            words.append(ref[i - 1]); path.append((i, j)); i -= 1; j -= 1
        elif T[i - 1, j] >= T[i, j - 1]:
            i -= 1
        else:
            j -= 1
    return words[::-1], path[::-1]

def rouge_l(cand, ref):
    T = lcs_table(ref, cand)
    return prf(T[-1, -1], len(cand), len(ref))

ref = tokenize(rouge_ref)
cand = tokenize(summaries["Перестановка"])
T = lcs_table(ref, cand)
print(pd.DataFrame(T, index=["∅"] + ref, columns=["∅"] + cand).to_string())
words, path = lcs_backtrack(T, ref, cand)
print(f"\nLCS = «{' '.join(words)}», длина {len(words)}")
P, R, F = rouge_l(cand, ref)
print(f"ROUGE-L: P = {len(words)}/{len(cand)} = {P:.3f}, R = {len(words)}/{len(ref)} = {R:.3f}, F1 = {F:.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(T, cmap="Blues", vmin=0, vmax=2 * T.max())
for i in range(T.shape[0]):
    for j in range(T.shape[1]):
        on_path = (i, j) in path
        ax.text(j, i, T[i, j], ha="center", va="center", fontsize=10,
                fontweight="bold" if on_path else "normal", color="#C94B2B" if on_path else "black")
        if on_path:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="#C94B2B", linewidth=2))
ax.set_xticks(range(len(cand) + 1)); ax.set_xticklabels(["∅"] + cand, rotation=40, ha="right")
ax.set_yticks(range(len(ref) + 1)); ax.set_yticklabels(["∅"] + ref)
ax.set_title("Таблица LCS: строки — эталон, столбцы — реферат")
plt.tight_layout()
plt.show()

Теперь все примеры лаборатории. Готовой функции ROUGE в nltk нет, поэтому LCS проверим независимым способом. Если разрешены только вставки и удаления (замена стоит 2, как удаление плюс вставка), редакционное расстояние равно $|a| + |b| - 2\cdot LCS$. Значит, `nltk.edit_distance(..., substitution_cost=2)` даёт LCS другим алгоритмом (тема 11).

In [ ]:
from nltk import edit_distance

rows = []
for name, text in summaries.items():
    cand = tokenize(text)
    p1, r1, f1 = rouge_n(cand, ref, 1)
    _, _, f2 = rouge_n(cand, ref, 2)
    pl, rl, fl = rouge_l(cand, ref)
    lcs = lcs_table(ref, cand)[-1, -1]
    d = edit_distance(ref, cand, substitution_cost=2)   # только вставки и удаления
    rows.append({"пример": name, "R1 P": p1, "R1 R": r1, "R1 F": f1, "R2 F": f2, "RL P": pl, "RL R": rl, "RL F": fl,
                 "LCS": lcs, "LCS по nltk.edit_distance": (len(ref) + len(cand) - d) // 2})
print(pd.DataFrame(rows).set_index("пример").round(3).to_string())

«Обратный порядок» получает ROUGE-1 = 1 при бессмысленном тексте, а ROUGE-L его разоблачает. «Длинный» реферат содержит весь эталон: полнота 1, точность ниже. «Перефраз» теряет из-за форм слов: «новая» и «новую», «библиотека» и «библиотеку» — разные токены.

### В стандартных инструментах (только Colab)

Стандартный пакет `rouge-score` (Google) по умолчанию оставляет в тексте только латинские буквы и цифры. Русский текст для него пуст, а узбекское oʻqidi распадается на o и qidi. Ячейка показывает это и исправляет, передав свой токенизатор.

In [ ]:
# Только для Colab: пакет rouge-score (Google)
!pip install -q rouge-score
try:
    from rouge_score import rouge_scorer
    text = summaries["Перестановка"]
    default = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"])
    from rouge_score import tokenizers
    print("токенизатор по умолчанию:", tokenizers.DefaultTokenizer().tokenize(rouge_ref))
    print("токенизатор по умолчанию:", tokenizers.DefaultTokenizer().tokenize("Oʻqituvchimiz kitob oʻqidi"))
    print("   ", {k: round(v.fmeasure, 3) for k, v in default.score(rouge_ref, text).items()})

    class OurTokenizer:                      # rouge-score принимает любой объект с методом tokenize
        def tokenize(self, s):
            return tokenize(s)

    fixed = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=OurTokenizer())
    for name, text in summaries.items():
        scores = fixed.score(rouge_ref, text)    # порядок аргументов: (эталон, реферат)
        cand = tokenize(text)
        ours = (rouge_n(cand, ref, 1)[2], rouge_n(cand, ref, 2)[2], rouge_l(cand, ref)[2])
        print(f"{name:<18} rouge-score: {[round(scores[k].fmeasure, 3) for k in ['rouge1', 'rouge2', 'rougeL']]}   наш: {[round(x, 3) for x in ours]}")
except Exception as e:
    print("Не получилось (нужен интернет и пакет rouge-score):", e)

## 3. Точность, полнота и F1 для именованных сущностей

В NER (тема 9) сравнивают множества сущностей: эталон (gold) и ответ модели (predicted). TP — найденные верно, FP — лишние, FN — пропущенные:
$P = \dfrac{TP}{TP+FP}$, $R = \dfrac{TP}{TP+FN}$, $F_1 = \dfrac{2PR}{P+R}$.

Что значит «верно»? При точном сравнении (как в CoNLL) должны совпасть и границы, и тип. При частичном достаточно пересечения с эталонной сущностью того же типа. Возьмём предложение из лаборатории: персону модель нашла правильно, у организации ошиблась границей, место назвала организацией и добавила лишнюю сущность. Обратите внимание: при точном сравнении ошибка границы стоит дважды, это и FP, и FN.

In [ ]:
tokens = ["Вчера", "Анна", "Сергеевна", "Петрова", "из", "Высшей", "школы", "экономики",
          "выступила", "на", "форуме", "в", "Нижнем", "Новгороде"]
# сущность = (номер предложения, первый токен, последний токен, тип)
gold = [(0, 1, 3, "PER"), (0, 5, 7, "ORG"), (0, 12, 13, "LOC")]
pred = [(0, 1, 3, "PER"),    # верно
        (0, 6, 7, "ORG"),    # граница: «школы экономики» без «Высшей»
        (0, 12, 13, "ORG"),  # тип: место названо организацией
        (0, 10, 10, "ORG")]  # лишняя: «форуме»

def show_entity(e, toks):
    return f"[{' '.join(toks[e[1]:e[2] + 1])}]{e[3]}"

def matches(p, g, mode):
    if mode == "exact":                                    # совпадают границы и тип
        return p == g
    same_sent_type = p[0] == g[0] and p[3] == g[3]         # partial: тот же тип и есть пересечение
    return same_sent_type and p[1] <= g[2] and g[1] <= p[2]

def ner_prf(gold, pred, mode="exact", verbose=False, toks=None):
    used, tp = set(), 0
    for p in pred:
        hit = next((g for g in gold if g not in used and matches(p, g, mode)), None)
        if hit is not None:
            used.add(hit); tp += 1
        if verbose:
            print(f"   {show_entity(p, toks):<34} -> {'TP' if hit else 'FP'}")
    if verbose:
        for g in gold:
            if g not in used:
                print(f"   {show_entity(g, toks):<34} -> FN (пропущена)")
    fp, fn = len(pred) - tp, len(gold) - tp
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return tp, fp, fn, P, R, F

for mode, name in [("exact", "точное"), ("partial", "частичное")]:
    print(f"=== {name}")
    tp, fp, fn, P, R, F = ner_prf(gold, pred, mode, verbose=True, toks=tokens)
    print(f"   TP = {tp}, FP = {fp}, FN = {fn};  P = {P:.3f}, R = {R:.3f}, F1 = {F:.3f}\n")

### Уровень сущностей и уровень токенов

На корпусе из шести предложений в разметке BIO сравним два уровня: по сущностям (точное совпадение) и по токенам (у каждого токена метка его типа, O не оценивается). Ошибка границы на уровне токенов стоит одного токена, а на уровне сущностей целой сущности, поэтому F1 по токенам почти всегда выше. По каждому типу видно, где модель ошибается.

Сверяем со `sklearn.metrics.precision_recall_fscore_support`. Для уровня сущностей строим пары (эталон, прогноз) по объединению множеств сущностей: найденной без пары сущности соответствует «O» в эталоне (FP), пропущенной — «O» в прогнозе (FN).

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

ner_corpus = [   # (предложение, эталонные метки BIO, метки модели)
    ("Мария Иванова переехала из Казани в Москву",
     "B-PER I-PER O O B-LOC O B-LOC", "B-PER I-PER O O B-LOC O B-LOC"),
    ("Студенты Национального университета Узбекистана поехали в Бухару",
     "O B-ORG I-ORG I-ORG O O B-LOC", "O B-ORG I-ORG B-LOC O O B-LOC"),
    ("Профессор Пётр Смирнов работает в Московском государственном университете",
     "O B-PER I-PER O O B-ORG I-ORG I-ORG", "B-PER I-PER I-PER O O B-ORG I-ORG I-ORG"),
    ("Вчера в Хиве прошёл фестиваль",
     "O O B-LOC O O", "O O B-ORG O O"),
    ("Студенты Самаркандского университета поехали в Навои",
     "O B-ORG I-ORG O O B-LOC", "O B-LOC O O O B-LOC"),
    ("Ольга и Илья приехали в Ташкент",
     "B-PER O B-PER O O B-LOC", "B-PER I-PER I-PER O O B-LOC"),
]

def bio_entities(tags, sent_id):
    ents, start, typ = [], None, None
    for i, t in enumerate(tags + ["O"]):
        # сущность кончается на O, на B- и на I- другого типа
        if t == "O" or t.startswith("B-") or t[2:] != typ:
            if start is not None:
                ents.append((sent_id, start, i - 1, typ))
            start, typ = (i, t[2:]) if t != "O" else (None, None)
    return ents

gold_ents = [e for k, (s, g, p) in enumerate(ner_corpus) for e in bio_entities(g.split(), k)]
pred_ents = [e for k, (s, g, p) in enumerate(ner_corpus) for e in bio_entities(p.split(), k)]
types = ["LOC", "ORG", "PER"]

def per_type_table(counts):
    rows = {}
    for t, (tp, fp, fn) in counts.items():
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        rows[t] = [tp, fp, fn, P, R, 2 * P * R / (P + R) if P + R else 0.0]
    df = pd.DataFrame(rows, index=["TP", "FP", "FN", "P", "R", "F1"]).T
    TP, FP, FN = df["TP"].sum(), df["FP"].sum(), df["FN"].sum()
    micro_P, micro_R = TP / (TP + FP), TP / (TP + FN)
    df.loc["micro"] = [TP, FP, FN, micro_P, micro_R, 2 * micro_P * micro_R / (micro_P + micro_R)]
    df.loc["macro"] = [np.nan, np.nan, np.nan] + list(df.loc[types, ["P", "R", "F1"]].mean())
    return df

# уровень сущностей: точное совпадение
ent_counts = {t: (len([e for e in pred_ents if e[3] == t and e in gold_ents]),
                  len([e for e in pred_ents if e[3] == t and e not in gold_ents]),
                  len([e for e in gold_ents if e[3] == t and e not in pred_ents])) for t in types}
# уровень токенов: метка без префикса B-/I-
y_true = [t.split("-")[-1] for s, g, p in ner_corpus for t in g.split()]
y_pred = [t.split("-")[-1] for s, g, p in ner_corpus for t in p.split()]
tok_counts = {t: (sum(a == t and b == t for a, b in zip(y_true, y_pred)),
                  sum(a != t and b == t for a, b in zip(y_true, y_pred)),
                  sum(a == t and b != t for a, b in zip(y_true, y_pred))) for t in types}

ent_table, tok_table = per_type_table(ent_counts), per_type_table(tok_counts)
print("=== по сущностям\n" + ent_table.round(3).to_string(na_rep=""))
print("\n=== по токенам\n" + tok_table.round(3).to_string(na_rep=""))

# проверка через sklearn; для сущностей — пары (эталон, прогноз), несовпавшей сущности соответствует "O"
union = sorted(set(gold_ents) | set(pred_ents))
ent_true = [e[3] if e in gold_ents else "O" for e in union]
ent_pred = [e[3] if e in pred_ents else "O" for e in union]
print()
for level, (yt, yp), table in [("по сущностям", (ent_true, ent_pred), ent_table), ("по токенам", (y_true, y_pred), tok_table)]:
    for avg in ["micro", "macro"]:
        P, R, F, _ = precision_recall_fscore_support(yt, yp, labels=types, average=avg, zero_division=0)
        ok = np.allclose([P, R, F], table.loc[avg, ["P", "R", "F1"]].astype(float))
        print(f"sklearn {level:<18} {avg}: P = {P:.3f}, R = {R:.3f}, F1 = {F:.3f}   {'совпадает ✓' if ok else '✗'}")

### Micro и macro

Когда классов несколько, F1 усредняют двумя способами:

- **macro-F1** — простое среднее F1 классов, редкий класс весит столько же, сколько частый (в sklearn это среднее F1, а не F1 от средних P и R);
- **micro-F1** — сначала складываем TP, FP и FN всех классов, потом считаем одну F1, частые классы весят больше. Если у каждого примера ровно один ответ, micro-F1 равна accuracy: каждая ошибка одновременно FP одного класса и FN другого.

Матрицы ошибок взяты из лаборатории (классификатор новостей по темам): строки — правильный класс, столбцы — ответ модели.

In [ ]:
from sklearn.metrics import f1_score, classification_report

classes = ["спорт", "экономика", "культура"]
presets = {
    "Сбалансированные классы": [[18, 1, 1], [2, 16, 2], [1, 2, 17]],
    "Редкий класс": [[50, 2, 1], [3, 30, 1], [3, 2, 1]],
    "Всегда «спорт»": [[55, 0, 0], [35, 0, 0], [5, 0, 0]],
}

def f1_from_matrix(M):
    M = np.array(M)
    tp = np.diag(M)
    fp = M.sum(axis=0) - tp          # остальные клетки столбца
    fn = M.sum(axis=1) - tp          # остальные клетки строки
    P = np.divide(tp, tp + fp, out=np.zeros(len(tp)), where=(tp + fp) > 0)   # 0/0 считаем равным 0
    R = np.divide(tp, tp + fn, out=np.zeros(len(tp)), where=(tp + fn) > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(len(tp)), where=(P + R) > 0)
    micro_P = tp.sum() / (tp.sum() + fp.sum())
    micro_R = tp.sum() / (tp.sum() + fn.sum())
    return P, R, F, F.mean(), 2 * micro_P * micro_R / (micro_P + micro_R), tp.sum() / M.sum()

def expand(M):
    # из матрицы ошибок обратно в списки ответов для sklearn
    y_true, y_pred = [], []
    for i, row in enumerate(M):
        for j, n in enumerate(row):
            y_true += [classes[i]] * n
            y_pred += [classes[j]] * n
    return y_true, y_pred

summary = {}
for name, M in presets.items():
    P, R, F, macro, micro, acc = f1_from_matrix(M)
    yt, yp = expand(M)
    sk_macro = f1_score(yt, yp, labels=classes, average="macro", zero_division=0)
    sk_micro = f1_score(yt, yp, labels=classes, average="micro", zero_division=0)
    print(f"=== {name}")
    print(pd.DataFrame({"P": P, "R": R, "F1": F}, index=classes).round(3).to_string())
    print(f"macro-F1 = {macro:.3f} (sklearn {sk_macro:.3f})   micro-F1 = {micro:.3f} (sklearn {sk_micro:.3f})   accuracy = {acc:.3f}\n")
    summary[name] = {"micro-F1": micro, "macro-F1": macro}

print(classification_report(*expand(presets[list(presets)[1]]), labels=classes, digits=3, zero_division=0))

ax = pd.DataFrame(summary).T.plot.bar(figsize=(7, 3.5), color=["#2B55C9", "#C94B2B"], rot=0)
ax.set_ylim(0, 1.05); ax.set_title("Micro-F1 и macro-F1 на трёх матрицах ошибок")
plt.tight_layout()
plt.show()

## 4. Задания

1. **Свои переводы.** Для `ref1` и `ref2` придумайте два перевода: (а) понятный человеку, но с BLEU = 0 без сглаживания; (б) с высоким BLEU, но с противоположным смыслом (например, с отрицанием «не»). Посчитайте для них BLEU (с `method2`) и chrF. Какая метрика заметила проблему, а какая нет?
2. **chrF++.** В chrF++ к символьным n-граммам (n = 1..6) добавляют словесные униграммы и биграммы, а $P$ и $R$ усредняют по всем 8 порядкам. Напишите `chrf_pp` и сравните BLEU, chrF и chrF++ на примерах «Синоним» и «Окончание». В Colab сверьтесь с `sacrebleu.CHRF(word_order=2)`.
3. **ROUGE по основам.** Посчитайте ROUGE-1, ROUGE-2 и ROUGE-L для «Перефраза», обрезав каждое слово до первых 5 букв (грубая замена лемматизации, как в практикуме 1). Насколько выросли оценки? Повторите для узбекской пары из лаборатории: эталон «Toshkent hokimligi shahar markazida yangi kutubxona qurishga qaror qildi», реферат «Toshkentda shahar markazida yangi kutubxona quriladi».
4. **Ещё один способ сравнения сущностей.** Добавьте в `matches` режим `"boundary"`: тип не важен, границы должны совпасть точно. Посчитайте P, R и F1 на корпусе раздела 3 (`gold_ents`, `pred_ents`) в трёх режимах. Каких ошибок у модели больше: в типах или в границах?
5. **Micro против macro.** Задайте матрицу ошибок `my_matrix` (3×3, классы из раздела 3), у которой micro-F1 ≥ 0,9, а macro-F1 ≤ 0,6. Проверьте её через `f1_score` и посчитайте ещё `average="weighted"`. К какому из двух средних ближе weighted и почему? Затем запустите ячейку самопроверки.

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# самопроверка к заданию 5: запустите после решения
M = globals().get("my_matrix")
if M is None:
    print("Сначала задайте my_matrix в задании 5.")
else:
    _, _, _, macro, micro, _ = f1_from_matrix(M)
    print(("✓" if micro >= 0.9 else "✗"), f"micro-F1 ≥ 0,9: micro-F1 = {micro:.3f}")
    print(("✓" if macro <= 0.6 else "✗"), f"macro-F1 ≤ 0,6: macro-F1 = {macro:.3f}")